In [1]:
import sys; sys.version

'3.12.4 (tags/v3.12.4:8e8a4ba, Jun  6 2024, 19:30:16) [MSC v.1940 64 bit (AMD64)]'

In [2]:
import sys
import os
import importlib
import pandas as pd

def importar_local(nome_modulo):
    """Força a importação da versão local do módulo"""
    # Caminhos
    notebook_dir = os.getcwd()
    project_root = os.path.dirname(notebook_dir)
    src_path = os.path.join(project_root, 'src')
    
    # Remove o módulo do cache se já estiver carregado
    if nome_modulo in sys.modules:
        del sys.modules[nome_modulo]
    
    # Remove caminhos que possam ter a versão instalada
    sys.path = [p for p in sys.path if 'comparar_fundos_br' not in p]
    
    # Adiciona o caminho local com prioridade
    sys.path.insert(0, project_root)
    sys.path.insert(0, src_path)
    
    # Importa o módulo
    modulo = importlib.import_module(nome_modulo)
    
    print(f" Módulo carregado de: {modulo.__file__}")
    return modulo

# Usar
comp = importar_local('comparar_fundos_br')

pyettj 0.4.1
 Módulo carregado de: C:\Users\rrafa\OneDrive - BNDES\Área de Trabalho\comparar_fundos_br\src\comparar_fundos_br\__init__.py


In [3]:
f"http://dados.cvm.gov.br/dados/FI/DOC/INF_DIARIO/DADOS/inf_diario_fi_{2026:02d}{8:02d}.zip"

'http://dados.cvm.gov.br/dados/FI/DOC/INF_DIARIO/DADOS/inf_diario_fi_202608.zip'

In [10]:
informe_diario_fundos = comp.fundosbr(anos=2026, meses=8, cnpj="43984537000146", output_format="pandas")

In [4]:
informe_diario_fundos.columns

Index(['TP_FUNDO', 'CNPJ_FUNDO', 'ID_SUBCLASSE', 'VL_TOTAL', 'VL_QUOTA', 'VL_PATRIM_LIQ', 'CAPTC_DIA', 'RESG_DIA', 'NR_COTST'], dtype='str')

In [ ]:
cadastro = comp.get_cadastro_fundos(classe=comp.get_classes(),
                                    output_format='pandas')
cadastro.head()

In [ ]:
base = comp.mesclar_bases(cadastro,
                          informe_diario_fundos,
                          output_format = "pandas")
base.head()

In [ ]:
base[["Situacao", "Tributacao_Longo_Prazo", "Classificacao", "Tipo_Classe", "Classificacao_Anbima",
      "Indicador_Desempenho", "Permitido_Aplicacao_CemPorCento_Exterior", "Classe_ESG", "Publico_Alvo", "Exclusivo"]].head()

In [ ]:
base.columns

In [9]:
dados = comp.DadosFinanceiros(proxy=None)

data_inicio, data_fim = "2008-01-01", "2026-08-01"

ptax = dados.cambio_ptax(data_inicio, data_fim, compra=False)
acoes_etfs = dados.stocks(["PETR4", "VALE3", "IVVB11"], data_inicio, data_fim)
multi = dados.benchmarks(data_inicio, data_fim, benchmark=["CDI", "IBOV", "sp500"], metodo_cdi="bacen")

df_benchmarks = pd.concat([multi, acoes_etfs], axis=1).sort_index() #cuidado com missing values

In [7]:
multi = dados.benchmarks(data_inicio, data_fim, benchmark=["CDI", "IBOV", "sp500"], metodo_cdi="anbima")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [8]:
multi

,CDI,Retorno CDI,Retorno Acumulado CDI,IBOV,Retorno IBOV,Retorno Acumulado IBOV,SP500,Retorno SP500,Retorno Acumulado SP500
2008-01-02,1480.099070,NaN,NaN,62815.000000,NaN,NaN,1447.160034,NaN,NaN
2008-01-03,1480.724919,0.000423,0.000423,62892.000000,0.001226,0.001226,1447.160034,0.000000,0.000000
2008-01-04,1481.355554,0.000426,0.000849,61037.000000,-0.029495,-0.028305,1411.630005,-0.024552,-0.024552
2008-01-07,1482.037659,0.000460,0.001310,60772.000000,-0.004342,-0.032524,1416.180054,0.003223,-0.021407
2008-01-08,1482.703342,0.000449,0.001760,62081.000000,0.021540,-0.011685,1390.189941,-0.018352,-0.039367
...,...,...,...,...,...,...,...,...,...
2026-07-27,8786.919080,0.000525,4.936710,175335.000000,0.007429,1.791292,7413.180176,0.000162,4.122571
2026-07-28,8791.587863,0.000531,4.939864,176565.000000,0.007015,1.810873,7428.779785,0.002104,4.133351
2026-07-29,8796.273160,0.000533,4.943030,173885.000000,-0.015179,1.768208,7316.149902,-0.015161,4.055522
2026-07-30,8801.000522,0.000537,4.946224,177159.000000,0.018829,1.820330,7437.629883,0.016604,4.139466


In [5]:
import getpass
import comparar_fundos_br as comp

user = getpass.getuser().lower()
pwd = getpass.getpass(prompt="Senha proxy: ")
proxy = "proxy.inf.bndes.net"
porta = 8080
proxies = {
    "http": f"http://{user}:{pwd}@{proxy}:{porta}",
    "https": f"http://{user}:{pwd}@{proxy}:{porta}",
}

fip = comp.get_fip(2026, proxy=proxies)
fip[fip['CNPJ_FUNDO_CLASSE'].isin(['32.625.186/0001-60'])]

Senha proxy:  ········


In [11]:
#Informe apenas o ano para obter os dados de FIPs disponíveis
fip = comp.get_fip(2026, proxy=proxies)
fip

Finalizado em 0.01 minutos


,TP_FUNDO_CLASSE,CNPJ_FUNDO_CLASSE,DENOM_SOCIAL,DT_COMPTC,VL_PATRIM_LIQ,...,QT_COTA_SUBSCR_CLASSE,QT_COTA_INTEGR_CLASSE,VL_QUOTA_CLASSE,DIREITO_POLIT_CLASSE,DIREITO_ECON_CLASSE
0,CLASSES - FIP,06.033.235/0001-66,CRT FIP - MULTIESTRATÉGIA,2026-04-30,28866473.760000000000,...,103.00000000,103.00000000,280257.02679611,N,N
1,CLASSES - FIP,06.962.594/0001-06,INVESTIDORES INSTITUCIONAIS II - FUNDO DE INVE...,2026-04-30,3942703.810000000000,...,407203.02985557,407203.02981000,9.68240290,N,N
2,CLASSES - FIP,06.962.594/0001-06,INVESTIDORES INSTITUCIONAIS II - FUNDO DE INVE...,2026-04-30,3942703.806799980000,...,407203.02985557,407203.02981000,9.68240290,N,N
3,CLASSES - FIP,07.319.087/0001-03,ASCET I - FUNDO DE INVESTIMENTO EM PARTICIPAÇŐ...,2026-04-30,-280022.050000000000,...,727.05091020,727.05091020,-234.09452847,N,N
4,CLASSES - FIP,07.319.087/0001-03,ASCET I - FUNDO DE INVESTIMENTO EM PARTICIPAÇŐ...,2026-04-30,-280022.050000000000,...,317.81279210,317.81279210,-234.09451051,N,N
...,...,...,...,...,...,...,...,...,...,...,...
2828,FIP,15.807.807/0001-08,VINCI REAL ESTATE FUNDO DE INVESTIMENTO EM PAR...,2026-04-30,4884449.770000000000,...,407615.00000000,196086.76335822,24.90963536,N,N
2829,FIP,39.976.918/0001-06,OCA FUNDO DE INVESTIMENTO EM PARTICIPAÇŐES EM ...,2026-04-30,44166648.860000000000,...,34035.18914000,34076.11160000,999.36095614,N,S
2830,FIP,39.976.918/0001-06,OCA FUNDO DE INVESTIMENTO EM PARTICIPAÇŐES EM ...,2026-04-30,44166648.860000000000,...,10118.77974000,10118.77974000,999.36095555,N,S
2831,FIP,44.544.175/0001-35,FRANCESCO FUNDO DE INVESTIMENTO EM PARTICIPAÇŐ...,2026-04-30,14590406.020000000000,...,152069.13000000,139939.70261653,104.26209108,N,N


In [18]:
fip.columns

Index(['TP_FUNDO_CLASSE', 'CNPJ_FUNDO_CLASSE', 'DENOM_SOCIAL', 'DT_COMPTC', 'VL_PATRIM_LIQ', 'QT_COTA', 'VL_PATRIM_COTA', 'NR_COTST', 'ENTID_INVEST', 'PUBLICO_ALVO', 'VL_CAP_COMPROM', 'QT_COTA_SUBSCR', 'VL_CAP_SUBSCR', 'QT_COTA_INTEGR', 'VL_CAP_INTEGR', 'VL_INVEST_FIP_COTA', 'NR_COTST_SUBSCR_PF', 'PR_COTA_SUBSCR_PF', 'NR_COTST_SUBSCR_PJ_NAO_FINANC', 'PR_COTA_SUBSCR_PJ_NAO_FINANC', 'NR_COTST_SUBSCR_BANCO', 'PR_COTA_SUBSCR_BANCO', 'NR_COTST_SUBSCR_CORRETORA_DISTRIB', 'PR_COTA_SUBSCR_CORRETORA_DISTRIB', 'NR_COTST_SUBSCR_PJ_FINANC', 'PR_COTA_SUBSCR_PJ_FINANC', 'NR_COTST_SUBSCR_INVNR', 'PR_COTA_SUBSCR_INVNR', 'NR_COTST_SUBSCR_EAPC', 'PR_COTA_SUBSCR_EAPC', 'NR_COTST_SUBSCR_EFPC', 'PR_COTA_SUBSCR_EFPC', 'NR_COTST_SUBSCR_RPPS', 'PR_COTA_SUBSCR_RPPS', 'NR_COTST_SUBSCR_SEGUR', 'PR_COTA_SUBSCR_SEGUR', 'NR_COTST_SUBSCR_CAPITALIZ', 'PR_COTA_SUBSCR_CAPITALIZ', 'NR_COTST_SUBSCR_FII', 'PR_COTA_SUBSCR_FII', 'NR_COTST_SUBSCR_FI', 'PR_COTA_SUBSCR_FI', 'NR_COTST_SUBSCR_DISTRIB', 'PR_COTA_SUBSCR_DISTRIB',


In [19]:
fip[fip['CNPJ_FUNDO_CLASSE'].isin(['32.625.186/0001-60'])]#.to_excel('exemplo_dados_fips_cvm.xlsx')

In [6]:
comp.get_fidc(2026, tabela = 'X', subtabela = 3, proxy=proxies)

Baixando meses:  67%|████████████████████████████████████                  | 8/12 [00:06<00:02,  1.48it/s]

 Mês 08 não disponível (status 404)


Baixando meses:  75%|████████████████████████████████████████▌             | 9/12 [00:06<00:01,  1.78it/s]

 Mês 09 não disponível (status 404)


Baixando meses:  83%|████████████████████████████████████████████▏        | 10/12 [00:07<00:01,  2.00it/s]

...

Baixando meses:  92%|████████████████████████████████████████████████▌    | 11/12 [00:07<00:00,  2.04it/s]

 Mês 11 não disponível (status 404)


Baixando meses: 100%|█████████████████████████████████████████████████████| 12/12 [00:08<00:00,  1.46it/s]

 Mês 12 não disponível (status 404)


,TP_FUNDO_CLASSE,CNPJ_FUNDO_CLASSE,DENOM_SOCIAL,DT_COMPTC,TAB_X_CLASSE_SERIE,TAB_X_VL_RENTAB_MES
0,Classe,05.754.060/0001-13,CATERPILLAR FUNDO DE INVESTIMENTO EM DIREITOS ...,2026-01-31,Subclasse Senior Subclasse 1,1.42
1,Classe,05.754.060/0001-13,CATERPILLAR FUNDO DE INVESTIMENTO EM DIREITOS ...,2026-01-31,Subclasse Subordinada |,0.00
2,Classe,06.018.364/0001-85,FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS ...,2026-01-31,Subclasse Senior Série 1,0.32
3,Classe,06.018.364/0001-85,FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS ...,2026-01-31,Subclasse Subordinada Subordinada 1 |,0.00
4,Classe,06.081.379/0001-98,KOL FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓR...,2026-01-31,Subclasse Senior Subclasse 1,1.19
...,...,...,...,...,...,...
83103,Fundo,63.212.949/0001-75,GENIAL SPECIAL SITUATIONS I FUNDO DE INVESTIME...,2026-07-31,Subclasse Subordinada |,0.00
83104,Fundo,65.953.190/0001-07,VERIDIUM STRUCTURED FUNDO DE INVESTIMENTO EM D...,2026-07-31,Subclasse Senior,0.00
83105,Fundo,65.953.190/0001-07,VERIDIUM STRUCTURED FUNDO DE INVESTIMENTO EM D...,2026-07-31,Subclasse Subordinada Subordinada 1 |,1.89
83106,Fundo,66.814.472/0001-96,RIZA KRATOS FUNDO DE INVESTIMENTO EM DIREITOS ...,2026-07-31,Subclasse Senior Série 1,0.00
